### delta rule

- 把状态 $S$ 看成一个 key → value 映射
    - 单头情况下：$S\in\mathbb R^{d_k\times d_v}$
    - 输入一个 key $k\in\mathbb R^{d_k}$，状态给出的 value 预测是：$\hat v=S^\top k\in\mathbb R^{d_v}$
    - 可以把 $S^\top$ 想成一个在线更新的小型线性层：$k\overset{S^\top}{\longmapsto}\hat v$
    - KDA 希望当前 key $k_t$ 对应当前 value $v_t$：$S_t^\top k_t\approx v_t$
- 最朴素的写入有什么问题
    - 最简单的关联记忆写法是 Hebbian update：$S_t=S_{t-1}+\beta_tk_tv_t^\top$
        - 其中 $k_tv_t^\top$ 是一个外积矩阵。
    - 但如果同一个 key 重复出现，value 会不停累加。例如：
        - 记忆已经知道 $k\mapsto v$；
        - 再执行一次 $S\leftarrow S+kv^\top$；
        - 查询结果可能从 $v$ 变成 $2v$。
        - 所以真正应该写入的不是完整的 $v_t$，而是记忆目前还没有学会的残差。
- Delta rule：先预测，再写误差
    - 首先查询当前旧状态：$\hat v_t=S_{t-1}^\top k_t$
    - 然后计算预测误差：$\delta_t=v_t-\hat v_t$
    - Delta rule 写入：$\boxed{S_t=S_{t-1}+\beta_tk_t\delta_t^\top}$
        - 代入 $\delta_t$：$S_t=S_{t-1}+\beta_tk_t\left(v_t-S_{t-1}^\top k_t\right)^\top$
     
$$
\begin{split}
S_t&=S_{t-1}+\beta_tk_t\left(v_t-S_{t-1}^\top k_t\right)^\top\\
&=S_{t-1}+\beta_tk_tv_t^\top-\beta_tk_tk_t^\top S_{t-1}\\
&=\left(I-\beta_tk_tk_t^\top\right)S_{t-1}+\beta_tk_tv_t^\top
\end{split}
$$

$I-\beta kk^\top$ 并不是凭空设计出来的复杂矩阵，它只是“先减掉旧预测，再加上新目标”展开后的结果。

#### $\beta$ 为什么可以解释成写入强度

KDA 会对 $k_t$ 做 L2 归一化：$k_t^\top k_t=1$

令旧预测：$\hat v_t=S_{t-1}^\top k_t$ 
更新后，再使用同一个 $k_t$ 查询：

$$
\begin{aligned}
S_t^\top k_t
&=
\hat v_t
+\beta_t\left(v_t-\hat v_t\right)(k_t^\top k_t)\\
&=
\hat v_t+\beta_t(v_t-\hat v_t)\\
&=
(1-\beta_t)\hat v_t+\beta_tv_t
\end{aligned}
$$

- $\beta_t=0$：完全不修改，结果仍是 $\hat v_t$；
- $\beta_t=0.5$：新预测移动到旧预测和目标之间的中点；
- $\beta_t=1$：新预测恰好变成 $v_t$。

所以 $\beta_t$ 就是“把预测往目标方向修正多少”。

### 引入 $\alpha$

$\alpha$ 不是从 Delta rule 的平方误差中必然推导出来的；它是 KDA 额外引入的、可学习的遗忘机制。做法是先用 $\alpha$ 衰减旧状态，再把“衰减后的状态”作为 Delta rule 的起点。

- KDA 不是直接对 $S_{t-1}$ 使用 Delta rule，而是先做逐通道衰减：$\bar S_t=\operatorname{Diag}(\alpha_t)S_{t-1}$
    - $S_{t-1}$ 的每行一个保留/遗忘率
    - $\text{row}_j(\bar S_{t})=\alpha_{t,j}\text{row}_j(S_{t-1})$
- 然后基于衰减后的状态计算预测：$\hat v_t=\bar S_t^\top k_t$ 
- 最后执行 Delta rule：$S_t=\bar S_t+\beta_tk_t\left(v_t-\bar S_t^\top k_t\right)^\top$
- 展开即得到论文公式：

$$
S_t
=
\left(I-\beta_tk_tk_t^\top\right)
\operatorname{Diag}(\alpha_t)S_{t-1}
+\beta_tk_tv_t^\top
$$

所以完整过程是：
$$
\boxed{
S_{t-1}
\xrightarrow{\alpha_t\text{ 衰减旧记忆}}
\bar S_t
\xrightarrow{k_t\text{ 查询}}
\hat v_t
\xrightarrow{v_t-\hat v_t\text{ 算误差}}
\delta_t
\xrightarrow{\beta_tk_t\delta_t^\top\text{ 写回}}
S_t
}
$$